In [1]:
import pandas as pd
import numpy as np

print("Loading raw datasets from the 'raw_data' folder...")

# Load all 6 tables into pandas dataframes
customers = pd.read_csv("raw_data/data/customers.csv")
locations = pd.read_csv("raw_data/data/locations.csv")
services = pd.read_csv("raw_data/data/services.csv")
channels = pd.read_csv("raw_data/data/channels.csv")
transactions = pd.read_csv("raw_data/data/transactions.csv")
support = pd.read_csv("raw_data/data/support_interactions.csv")

print("\nDataset Shapes Loaded Successfully:")
print(f"• Customers: {customers.shape[0]:,} rows x {customers.shape[1]} columns")
print(f"• Locations: {locations.shape[0]:,} rows x {locations.shape[1]} columns")
print(f"• Services: {services.shape[0]:,} rows x {services.shape[1]} columns")
print(f"• Channels: {channels.shape[0]:,} rows x {channels.shape[1]} columns")
print(f"• Transactions: {transactions.shape[0]:,} rows x {transactions.shape[1]} columns")
print(f"• Support Interactions: {support.shape[0]:,} rows x {support.shape[1]} columns")


Loading raw datasets from the 'raw_data' folder...

Dataset Shapes Loaded Successfully:
• Customers: 10,000 rows x 8 columns
• Locations: 10 rows x 4 columns
• Services: 8 rows x 4 columns
• Channels: 5 rows x 3 columns
• Transactions: 155,079 rows x 12 columns
• Support Interactions: 20,025 rows x 9 columns


In [3]:
# Create a reusable function to report missing data simply
def check_missing(df, name):
    print(f"\nMissing Data Report for: {name.upper()}")
    missing_count = df.isna().sum()
    missing_percent = (df.isna().mean() * 100).round(2)
    
    # Combine counts and percentages into a neat table
    report = pd.DataFrame({
        'Missing Rows': missing_count,
        'Percentage (%)': missing_percent
    })
    # Only show columns that actually have missing data to keep it clean
    print(report[report['Missing Rows'] > 0])

# Run the check on our main tables
check_missing(customers, "Customers")
check_missing(transactions, "Transactions")
check_missing(support, "Support Interactions")



Missing Data Report for: CUSTOMERS
                     Missing Rows  Percentage (%)
occupation_category           638            6.38
acquisition_channel           417            4.17

Missing Data Report for: TRANSACTIONS
                         Missing Rows  Percentage (%)
failure_reason                 139939           90.24
processing_time_seconds          2372            1.53

Missing Data Report for: SUPPORT INTERACTIONS
                 Missing Rows  Percentage (%)
transaction_id           4006           20.00
customer_rating          4422           22.08


In [4]:
print("Checking unique text values before cleaning...\n")

print("1. Unique Segments in Customers Table:")
print(customers['customer_segment'].unique())

print("\n2. Unique Device Types in Transactions Table:")
print(transactions['device_type'].unique())

print("\n3. Unique Support Channels in Support Table:")
print(support['support_channel'].unique())


Checking unique text values before cleaning...

1. Unique Segments in Customers Table:
['Affluent' 'Emerging' 'Mass Market' 'High Value' 'Small Business'
 'affluent' 'AFFLUENT' 'mass market' 'Mass Market ' 'MASS MARKET'
 'high value' 'emerging' 'Affluent ' 'small business' 'High Value '
 'EMERGING' 'Emerging ' 'Small Business ' 'SMALL BUSINESS']

2. Unique Device Types in Transactions Table:
['POS Terminal' 'USSD' 'iOS' 'Android' 'Web/Desktop' 'Android ' 'IOS'
 'ANDROID' 'POS TERMINAL' 'pos terminal' 'android' 'ussd' 'WEB/DESKTOP'
 'Web/Desktop ' 'ios' 'POS Terminal ' 'USSD ' 'web/desktop' 'iOS ']

3. Unique Support Channels in Support Table:
['Email' 'Phone' 'Live Chat' 'In-App Support' 'Social Media' 'phone'
 'LIVE CHAT' 'in-app support' 'Social Media ' 'live chat' 'Live Chat '
 'EMAIL' 'PHONE' 'email' 'IN-APP SUPPORT' 'Phone ' 'Email ' 'social media'
 'In-App Support ' 'SOCIAL MEDIA']


In [5]:
print("Standardizing text categories and cleaning white spaces...\n")

# 1. Clean the Customers Table
customers['customer_segment'] = customers['customer_segment'].astype(str).str.strip().str.title()

# 2. Clean the Transactions Table (and fix specific tech acronyms like USSD and iOS)
transactions['device_type'] = transactions['device_type'].astype(str).str.strip().str.title()
transactions['device_type'] = transactions['device_type'].replace({
    'Ussd': 'USSD',
    'Ios': 'iOS',
    'Pos Terminal': 'POS Terminal'
})

# 3. Clean the Support Interactions Table
support['support_channel'] = support['support_channel'].astype(str).str.strip().str.title()

print("Text categorization successfully cleaned! Let's verify the results:\n")

print("Cleaned Customer Segments:", customers['customer_segment'].unique())
print("Cleaned Device Types:     ", transactions['device_type'].unique())
print("Cleaned Support Channels: ", support['support_channel'].unique())


Standardizing text categories and cleaning white spaces...

Text categorization successfully cleaned! Let's verify the results:

Cleaned Customer Segments: ['Affluent' 'Emerging' 'Mass Market' 'High Value' 'Small Business']
Cleaned Device Types:      ['POS Terminal' 'USSD' 'iOS' 'Android' 'Web/Desktop']
Cleaned Support Channels:  ['Email' 'Phone' 'Live Chat' 'In-App Support' 'Social Media']


In [6]:
print("Auditing possible duplicates (time-aware)...\n")

transaction_signature = ['customer_id', 'service_id', 'location_id', 'channel_id', 'transaction_amount', 'transaction_status']

# Sort by signature and time, then measure the gap between matching rows.
tx = transactions.copy()
tx['transaction_datetime'] = pd.to_datetime(tx['transaction_datetime'])
tx = tx.sort_values(transaction_signature + ['transaction_datetime'])
gap = tx.groupby(transaction_signature)['transaction_datetime'].diff()

# A repeat within 5 minutes looks like a resubmission; a repeat months apart is normal behaviour.
tx['is_near_duplicate'] = gap <= pd.Timedelta(minutes=5)

same_signature_any_date = transactions.duplicated(subset=transaction_signature, keep='first').sum()
print(f"• Same signature on any date:      {same_signature_any_date} rows")
print(f"• Same signature within 5 minutes: {tx['is_near_duplicate'].sum()} rows")

# Support tickets: only count rows identical in every column except the ID.
support_cols = [c for c in support.columns if c != 'interaction_id']
print(f"• Support exact copies:            {support.duplicated(subset=support_cols).sum()} rows")

Auditing possible duplicates (time-aware)...

• Same signature on any date:      704 rows
• Same signature within 5 minutes: 58 rows
• Support exact copies:            25 rows


In [7]:
print("Keeping all rows, flagging near-duplicates for transparency...\n")

transactions_cleaned = transactions.copy()
transactions_cleaned['is_near_duplicate'] = tx['is_near_duplicate'].reindex(transactions_cleaned.index).fillna(False)

support_cleaned = support.copy()
support_cleaned['is_exact_copy'] = support.duplicated(subset=support_cols)

print(f"• Transactions kept: {len(transactions_cleaned):,} ({transactions_cleaned['is_near_duplicate'].sum()} flagged)")
print(f"• Support tickets kept: {len(support_cleaned):,} ({support_cleaned['is_exact_copy'].sum()} flagged)")

# Sensitivity check: does removing the flagged rows change the story?
kept = transactions_cleaned[~transactions_cleaned['is_near_duplicate']]
print(f"\nFailure rate, all rows:     {(transactions_cleaned['transaction_status'] == 'Failed').mean() * 100:.2f}%")
print(f"Failure rate, without flags: {(kept['transaction_status'] == 'Failed').mean() * 100:.2f}%")

Keeping all rows, flagging near-duplicates for transparency...

• Transactions kept: 155,079 (58 flagged)
• Support tickets kept: 20,025 (25 flagged)

Failure rate, all rows:     9.52%
Failure rate, without flags: 9.52%


In [8]:
print("Parsing text date strings into real datetime objects...\n")

# 1. Parse date columns using pandas datetime tools
customers['registration_date'] = pd.to_datetime(customers['registration_date'], errors='coerce')
transactions_cleaned['transaction_datetime'] = pd.to_datetime(transactions_cleaned['transaction_datetime'], errors='coerce')
support_cleaned['interaction_datetime'] = pd.to_datetime(support_cleaned['interaction_datetime'], errors='coerce')

print("Data types converted. Let's verify the columns are now read as datetime64:")
print(f"• Customers Registration Date:     {customers['registration_date'].dtype}")
print(f"• Transactions Datetime:          {transactions_cleaned['transaction_datetime'].dtype}")
print(f"• Support Interactions Datetime:  {support_cleaned['interaction_datetime'].dtype}")

# 2. Check for data integrity: Find out our overall project timeline boundaries
print("\nVerified Project Operational Timeline:")
print(f"• System Transactions Start: {transactions_cleaned['transaction_datetime'].min()}")
print(f"• System Transactions End:   {transactions_cleaned['transaction_datetime'].max()}")


Parsing text date strings into real datetime objects...

Data types converted. Let's verify the columns are now read as datetime64:
• Customers Registration Date:     datetime64[ns]
• Transactions Datetime:          datetime64[ns]
• Support Interactions Datetime:  datetime64[ns]

Verified Project Operational Timeline:
• System Transactions Start: 2024-10-01 02:49:28
• System Transactions End:   2026-08-31 23:57:34


In [9]:
print("Scanning tables for numeric validation outliers and errors...\n")

# 1. Transactions Table checks
negative_amounts = (transactions_cleaned['transaction_amount'] <= 0).sum()
negative_processing = (transactions_cleaned['processing_time_seconds'] <= 0).sum()

# 2. Support Interactions Table checks
invalid_ratings = (~support_cleaned['customer_rating'].dropna().between(1, 5)).sum()
negative_resolution = (support_cleaned['resolution_time_minutes'] <= 0).sum()

print("Integrity Check Summary Result:")
print(f"• Transaction records with an amount <= 0:         {negative_amounts} rows")
print(f"• Transaction records with a processing time <= 0:  {negative_processing} rows")
print(f"• Support interactions with a rating outside 1-5:  {invalid_ratings} rows")
print(f"• Support interactions with a resolution time <= 0: {negative_resolution} rows")


Scanning tables for numeric validation outliers and errors...

Integrity Check Summary Result:
• Transaction records with an amount <= 0:         0 rows
• Transaction records with a processing time <= 0:  0 rows
• Support interactions with a rating outside 1-5:  0 rows
• Support interactions with a resolution time <= 0: 0 rows


In [10]:
print("Business Question 1: Extracting Overall Transaction Status Profile...")
# Calculate baseline value counts and percentages
status_counts = transactions_cleaned['transaction_status'].value_counts()
status_percents = (transactions_cleaned['transaction_status'].value_counts(normalize=True) * 100).round(2)

# Combine into a neat summary table
status_report = pd.DataFrame({
    'Total Transactions': status_counts,
    'Proportion (%)': status_percents
})
print(status_report)

Business Question 1: Extracting Overall Transaction Status Profile...
                    Total Transactions  Proportion (%)
transaction_status                                    
Success                         135767           87.55
Failed                           14761            9.52
Pending                           4551            2.93


In [11]:
print("Business Question 2: Isolating Device Types with the Highest Failure Rates...")
# Group by device type and look at the distribution of transaction statuses
device_failures = pd.crosstab(
    transactions_cleaned['device_type'], 
    transactions_cleaned['transaction_status'], 
    normalize='index'
).mul(100).round(2)

# Sort by failure percentage so the biggest problems float to the top
print(device_failures.sort_values(by='Failed', ascending=False))

Business Question 2: Isolating Device Types with the Highest Failure Rates...
transaction_status  Failed  Pending  Success
device_type                                 
POS Terminal         10.70     2.98    86.32
USSD                  9.73     3.06    87.21
Android               9.38     2.85    87.78
iOS                   9.05     2.97    87.98
Web/Desktop           8.81     3.05    88.14


In [12]:
print("Business Question 3: Finding the Locations with the Weakest Transaction Performance...\n")

# Calculate failure rates by location using a cross-tabulation table
location_failures = pd.crosstab(
    transactions_cleaned['location_id'], 
    transactions_cleaned['transaction_status'], 
    normalize='index'
).mul(100).round(2)

# Sort by failed transactions so we can see the worst performing regions first
print(location_failures.sort_values(by='Failed', ascending=False).head(5))


Business Question 3: Finding the Locations with the Weakest Transaction Performance...

transaction_status  Failed  Pending  Success
location_id                                 
LOC005               10.10     2.94    86.96
LOC009                9.97     2.97    87.06
LOC007                9.78     2.87    87.35
LOC004                9.73     2.77    87.50
LOC006                9.63     2.93    87.44


In [13]:
print("Business Question 4: Analyzing Monthly Transaction Performance Trends...\n")

# Step 1: Create a Year-Month column to group our timeline cleanly
transactions_cleaned['year_month'] = transactions_cleaned['transaction_datetime'].dt.to_period('M')

# Step 2: Cross-tabulate Year-Month against Transaction Status to see monthly failure percentages
monthly_trends = pd.crosstab(
    transactions_cleaned['year_month'], 
    transactions_cleaned['transaction_status'], 
    normalize='index'
).mul(100).round(2)

# Print the full monthly timeline report
print(monthly_trends)


Business Question 4: Analyzing Monthly Transaction Performance Trends...

transaction_status  Failed  Pending  Success
year_month                                  
2024-10               9.11     2.89    88.00
2024-11               9.96     3.03    87.01
2024-12               9.35     2.50    88.15
2025-01               9.74     2.92    87.34
2025-02               8.56     3.30    88.14
2025-03               8.86     2.73    88.41
2025-04               9.15     2.96    87.89
2025-05               9.56     2.71    87.73
2025-06              10.21     2.79    87.01
2025-07               9.46     2.67    87.87
2025-08              10.01     2.47    87.52
2025-09               9.94     3.25    86.81
2025-10               9.75     2.94    87.30
2025-11               9.09     2.89    88.02
2025-12               9.24     3.26    87.50
2026-01               9.10     2.97    87.93
2026-02              10.26     2.72    87.02
2026-03               9.66     2.95    87.39
2026-04               9.19

In [14]:
print("Business Question 5: Analyzing Financial Volume and Fees Collected by Customer Segment...\n")

# Step 1: Merge transactions with customer details on customer_id
tx_customer_merged = pd.merge(transactions_cleaned, customers, on='customer_id', how='inner')

# Step 2: Group by customer segment and calculate total volumes, total fees, and average transactions
segment_revenue = tx_customer_merged.groupby('customer_segment').agg(
    total_transaction_count=('transaction_id', 'count'),
    total_fees_collected_ngn=('fee_amount', 'sum'),
    avg_transaction_amount_ngn=('transaction_amount', 'mean')
).reset_index()

# Round out decimal columns to keep the financial ledger clean
segment_revenue['total_fees_collected_ngn'] = segment_revenue['total_fees_collected_ngn'].round(2)
segment_revenue['avg_transaction_amount_ngn'] = segment_revenue['avg_transaction_amount_ngn'].round(2)

# Print the final clean summary table sorted by total fees collected
print(segment_revenue.sort_values(by='total_fees_collected_ngn', ascending=False).to_string(index=False))


Business Question 5: Analyzing Financial Volume and Fees Collected by Customer Segment...

customer_segment  total_transaction_count  total_fees_collected_ngn  avg_transaction_amount_ngn
     Mass Market                    52622                1846016.58                    12279.75
        Emerging                    33669                1210206.79                    13036.80
  Small Business                    27294                 993606.54                    14925.70
        Affluent                    27340                 961483.83                    14091.12
      High Value                    14154                 505469.69                    14043.41


In [15]:
print("Business Question 6: Auditing System Performance and Failure Rates Across Customer Tiers...\n")

# Step 1: Use the previously merged 'tx_customer_merged' data to find failure counts by segment
segment_performance = tx_customer_merged.groupby('customer_segment').agg(
    total_tx=('transaction_id', 'count'),
    failed_tx=('transaction_status', lambda x: (x == 'Failed').sum()),
    avg_processing_speed_sec=('processing_time_seconds', 'mean')
).reset_index()

# Step 2: Calculating the exact failure rate percentage for each customer tier
segment_performance['failure_rate_percentage'] = (
    (segment_performance['failed_tx'] / segment_performance['total_tx']) * 100
).round(2)

segment_performance['avg_processing_speed_sec'] = segment_performance['avg_processing_speed_sec'].round(2)

# Print the report sorted by failure rate percentage
print(segment_performance.sort_values(by='failure_rate_percentage', ascending=False).to_string(index=False))


Business Question 6: Auditing System Performance and Failure Rates Across Customer Tiers...

customer_segment  total_tx  failed_tx  avg_processing_speed_sec  failure_rate_percentage
     Mass Market     52622       5105                      9.92                     9.70
  Small Business     27294       2641                      9.62                     9.68
        Emerging     33669       3155                      9.79                     9.37
        Affluent     27340       2551                      9.65                     9.33
      High Value     14154       1309                     10.27                     9.25


In [16]:
print("Busines Question 7: Identifying the Most Frequent Customer Support Issue Categories...\n")

# Calculating the total count of support tickets per category
support_counts = support_cleaned['support_category'].value_counts()

# Calculating the percentage distribution of support issues
support_percents = (support_cleaned['support_category'].value_counts(normalize=True) * 100).round(2)

# Combining counts and percentages into a neat summary ledger
support_report = pd.DataFrame({
    'Total Tickets': support_counts,
    'Proportion (%)': support_percents
})

print(support_report)


Busines Question 7: Identifying the Most Frequent Customer Support Issue Categories...

                     Total Tickets  Proportion (%)
support_category                                  
Failed Transaction            4255           21.25
Refund Request                2744           13.70
Account Issue                 2648           13.22
Delayed Transaction           2503           12.50
Card Issue                    2297           11.47
Payment Reversal              2083           10.40
Account Access                2008           10.03
Other                         1487            7.43


In [17]:
print("Business Question 8: Analyzing Customer Ratings and Response Times Across Transaction Statuses...\n")

# Step 1: Merging the support dataset with transactions on transaction_id
support_tx_link = pd.merge(support_cleaned, transactions_cleaned, on='transaction_id', how='inner')

# Step 2: Grouping by the status of the transaction and finding the average customer satisfaction and response speed
satisfaction_audit = support_tx_link.groupby('transaction_status').agg(
    total_tickets=('interaction_id', 'count'),
    avg_customer_rating=('customer_rating', 'mean'),
    avg_resolution_time_mins=('resolution_time_minutes', 'mean')
).reset_index()

# Rounding up numbers for a professional presentation
satisfaction_audit['avg_customer_rating'] = satisfaction_audit['avg_customer_rating'].round(2)
satisfaction_audit['avg_resolution_time_mins'] = satisfaction_audit['avg_resolution_time_mins'].round(1)

print(satisfaction_audit.to_string(index=False))


Business Question 8: Analyzing Customer Ratings and Response Times Across Transaction Statuses...

transaction_status  total_tickets  avg_customer_rating  avg_resolution_time_mins
            Failed           8406                 3.36                      67.6
           Pending           2605                 3.37                      73.6
           Success           5008                 3.63                      54.4


In [18]:
print("Business Question 9: Isolating extreme high-value transactions...\n")

amount = transactions_cleaned['transaction_amount']
print(f"• 99th percentile: ₦{amount.quantile(0.99):,.0f}")
print(f"• Above ₦30,000: {(amount > 30000).sum():,} (the top 1%, by definition)")

extreme = transactions_cleaned[amount > 1_000_000]
share = extreme['transaction_amount'].sum() / amount.sum() * 100
print(f"• Above ₦1,000,000: {len(extreme):,} transactions ({len(extreme) / len(amount) * 100:.2f}% of all)")
print(f"• Their share of total value: {share:.1f}%")
print(extreme[['transaction_id', 'customer_id', 'transaction_amount', 'transaction_status']]
      .sort_values('transaction_amount', ascending=False).head(5).to_string(index=False))

Business Question 9: Isolating extreme high-value transactions...

• 99th percentile: ₦29,953
• Above ₦30,000: 1,542 (the top 1%, by definition)
• Above ₦1,000,000: 240 transactions (0.15% of all)
• Their share of total value: 67.0%
transaction_id customer_id  transaction_amount transaction_status
   TXN00153549  CUST003837          9781493.84            Success
   TXN00149108  CUST009838          9718329.14            Success
   TXN00150878  CUST004698          9702641.43            Success
   TXN00098661  CUST004418          9689222.57            Success
   TXN00018330  CUST008071          9663746.84            Success


In [19]:
total = len(transactions_cleaned)
fail_rate = (transactions_cleaned['transaction_status'] == 'Failed').mean() * 100

by_channel = (transactions_cleaned.merge(channels, on='channel_id')
              .groupby('channel_name')['transaction_status']
              .apply(lambda s: (s == 'Failed').mean() * 100).sort_values(ascending=False))

by_city = (transactions_cleaned.merge(locations, on='location_id')
           .groupby('city')['transaction_status']
           .apply(lambda s: (s == 'Failed').mean() * 100).sort_values(ascending=False))

linked = support_cleaned.merge(transactions_cleaned[['transaction_id', 'transaction_status']],
                               on='transaction_id', how='inner')

by_status = linked.groupby('transaction_status').agg(
    rating=('customer_rating', 'mean'), minutes=('resolution_time_minutes', 'mean'))
failed_ticket_share = (support_cleaned['support_category'] == 'Failed Transaction').mean() * 100

print(f"""
{'=' * 80}
PULSEPAY DATA PROJECT: KEY FINDINGS (all numbers calculated from the data)
{'=' * 80}

1. FAILURES ARE HIGHEST ON THE PHYSICAL-AGENT CHANNELS
   • {fail_rate:.2f}% of {total:,} transactions fail.
   • By channel: {by_channel.index[0]} ({by_channel.iloc[0]:.2f}%), {by_channel.index[1]} ({by_channel.iloc[1]:.2f}%), {by_channel.index[2]} ({by_channel.iloc[2]:.2f}%).
   • By city: {by_city.index[0]} ({by_city.iloc[0]:.2f}%) and {by_city.index[1]} ({by_city.iloc[1]:.2f}%); the spread across cities is small.

2. FAILED TRANSACTIONS HURT SUPPORT AND SATISFACTION
   • 'Failed Transaction' is the biggest ticket category ({failed_ticket_share:.1f}% of tickets).
   • Rating: {by_status.loc['Success', 'rating']:.2f} stars (success) vs {by_status.loc['Failed', 'rating']:.2f} (failed).
   • Resolution: {by_status.loc['Success', 'minutes']:.1f} min vs {by_status.loc['Failed', 'minutes']:.1f} min; pending takes longest ({by_status.loc['Pending', 'minutes']:.1f} min).

3. A TINY GROUP OF VERY LARGE TRANSACTIONS CARRIES MOST OF THE VALUE
   • {len(extreme):,} transactions above ₦1,000,000 make up {share:.1f}% of total value.
   • Largest single transaction: ₦{amount.max():,.0f}. A risk team should review these.
{'=' * 80}
""")


PULSEPAY DATA PROJECT: KEY FINDINGS (all numbers calculated from the data)

1. FAILURES ARE HIGHEST ON THE PHYSICAL-AGENT CHANNELS
   • 9.52% of 155,079 transactions fail.
   • By channel: Agent (10.72%), POS (10.67%), USSD (9.76%).
   • By city: Benin City (10.10%) and Ilorin (9.97%); the spread across cities is small.

2. FAILED TRANSACTIONS HURT SUPPORT AND SATISFACTION
   • 'Failed Transaction' is the biggest ticket category (21.2% of tickets).
   • Rating: 3.63 stars (success) vs 3.36 (failed).
   • Resolution: 54.4 min vs 67.6 min; pending takes longest (73.6 min).

3. A TINY GROUP OF VERY LARGE TRANSACTIONS CARRIES MOST OF THE VALUE
   • 240 transactions above ₦1,000,000 make up 67.0% of total value.
   • Largest single transaction: ₦9,781,494. A risk team should review these.

